In [1]:
import torch
print("GPU available:", torch.cuda.is_available())

!pip install -q transformers datasets accelerate

import json
train = json.load(open("train.json"))
config = json.load(open("config.json"))
print("Training pairs:", len(train))
print("Settings:", config)

GPU available: True
Training pairs: 12
Settings: {'model': 'Flan-T5-small', 'epochs': 3, 'learning_rate': 0.0001, 'quantization': 'None'}


In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# 1. Load the student (Flan-T5-small)
model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to("cuda")

# 2. Study settings
lr = config["learning_rate"]
epochs = 20   # your data is tiny, so we study more times than the saved setting
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

# 3. Study loop
model.train()
for epoch in range(epochs):
    total_loss = 0
    for item in train:
        inputs = tokenizer(item["instruction"], return_tensors="pt").to("cuda")
        labels = tokenizer(item["output"], return_tensors="pt").input_ids.to("cuda")
        loss = model(**inputs, labels=labels).loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs}  mistakes score (loss): {total_loss/len(train):.3f}")

# 4. Test the student
model.eval()
def ask(question):
    inputs = tokenizer(question, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=40)
    return tokenizer.decode(out[0], skip_special_tokens=True)

print("\nTest:")
print("Q: What are your timings?")
print("A:", ask("What are your timings?"))

# 5. Save the trained model
model.save_pretrained("my_chatbot")
tokenizer.save_pretrained("my_chatbot")
print("Model saved!")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

... (4 outputs left out)

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Epoch 1/20  mistakes score (loss): 4.304
Epoch 2/20  mistakes score (loss): 3.434
Epoch 3/20  mistakes score (loss): 2.781
Epoch 4/20  mistakes score (loss): 2.514
Epoch 5/20  mistakes score (loss): 2.195
Epoch 6/20  mistakes score (loss): 1.954
Epoch 7/20  mistakes score (loss): 1.680
Epoch 8/20  mistakes score (loss): 1.342
Epoch 9/20  mistakes score (loss): 1.201
Epoch 10/20  mistakes score (loss): 1.033
Epoch 11/20  mistakes score (loss): 0.869
Epoch 12/20  mistakes score (loss): 0.635
Epoch 13/20  mistakes score (loss): 0.542
Epoch 14/20  mistakes score (loss): 0.621
Epoch 15/20  mistakes score (loss): 0.360
Epoch 16/20  mistakes score (loss): 0.240
Epoch 17/20  mistakes score (loss): 0.235
Epoch 18/20  mistakes score (loss): 0.185
Epoch 19/20  mistakes score (loss): 0.193
Epoch 20/20  mistakes score (loss): 0.194

Test:
Q: What are your timings?
A: The festival starts at 11:00 AM


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved!


In [3]:
print("Questions the model studied:\n")
for item in train[:5]:
    print("Q:", item["instruction"])
    print("Correct:", item["output"])
    print("Model said:", ask(item["instruction"]))
    print()

Questions the model studied:

Q: Is the shop open on holidays?
Correct: Only on festival days from 10 AM to 2 PM
Model said: Only on festival days from 10 AM to 2 PM

Q: Which days are you closed?
Correct: Sundays
Model said: Sundays

Q: Is there parking?
Correct: Yes free parking is available
Model said: Yes free parking is available

Q: What is your name?
Correct: I am the shop assistant.
Model said: I am the shop assistant.

Q: Where is the office?
Correct: Kozhikode
Model said: Kozhikode



In [5]:
!zip -r my_chatbot.zip my_chatbot
from google.colab import files
files.download("my_chatbot.zip")

updating: my_chatbot/ (stored 0%)
updating: my_chatbot/model.safetensors (deflated 7%)
updating: my_chatbot/config.json (deflated 63%)
updating: my_chatbot/tokenizer.json (deflated 75%)
updating: my_chatbot/generation_config.json (deflated 29%)
updating: my_chatbot/tokenizer_config.json (deflated 83%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>